Starting Gold processing

--- Gold Processing Summary ---
  - gold.sales_by_product: 500 rows
  - gold.revenue_by_customer: 9940 rows
  - gold.customer_segmentation: 4 rows
  - gold.daily_weekly_trends: 2679 rows

Gold processing completed.
Exit code: 0


In [0]:
%sql

-- ============================================================
-- GOLD FINAL RUNTIME VALIDATION
-- ============================================================

-- 1. Gold table row counts
SELECT
    'TABLE_ROW_COUNTS' AS validation,
    'sales_by_product' AS check_name,
    COUNT(*) AS actual_value,
    500 AS expected_value,
    CASE WHEN COUNT(*) = 500 THEN 'PASS' ELSE 'FAIL' END AS status
FROM gold.sales_by_product

UNION ALL

SELECT
    'TABLE_ROW_COUNTS',
    'revenue_by_customer',
    COUNT(*),
    9940,
    CASE WHEN COUNT(*) = 9940 THEN 'PASS' ELSE 'FAIL' END
FROM gold.revenue_by_customer

UNION ALL

SELECT
    'TABLE_ROW_COUNTS',
    'customer_segmentation',
    COUNT(*),
    4,
    CASE WHEN COUNT(*) = 4 THEN 'PASS' ELSE 'FAIL' END
FROM gold.customer_segmentation

UNION ALL

SELECT
    'TABLE_ROW_COUNTS',
    'daily_weekly_trends',
    COUNT(*),
    2679,
    CASE WHEN COUNT(*) = 2679 THEN 'PASS' ELSE 'FAIL' END
FROM gold.daily_weekly_trends

UNION ALL

-- 2. Product grain
SELECT
    'GRAIN',
    'Duplicate product_id',
    COUNT(*) - COUNT(DISTINCT product_id),
    0,
    CASE
        WHEN COUNT(*) = COUNT(DISTINCT product_id)
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM gold.sales_by_product

UNION ALL

-- 3. Customer grain
SELECT
    'GRAIN',
    'Duplicate customer_id',
    COUNT(*) - COUNT(DISTINCT customer_id),
    0,
    CASE
        WHEN COUNT(*) = COUNT(DISTINCT customer_id)
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM gold.revenue_by_customer

UNION ALL

-- 4. Valid segmentation values
SELECT
    'SEGMENTATION',
    'Invalid segment_type',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.customer_segmentation
WHERE segment_type NOT IN
(
    'High-Value',
    'Repeat',
    'One-Time',
    'Inactive'
)

UNION ALL

-- 5. Segmentation customer-count reconciliation
SELECT
    'SEGMENTATION',
    'Customer count reconciliation',
    (
        SELECT SUM(customer_count)
        FROM gold.customer_segmentation
    ),
    (
        SELECT COUNT(*)
        FROM gold.revenue_by_customer
    ),
    CASE
        WHEN
            (
                SELECT SUM(customer_count)
                FROM gold.customer_segmentation
            )
            =
            (
                SELECT COUNT(*)
                FROM gold.revenue_by_customer
            )
        THEN 'PASS'
        ELSE 'FAIL'
    END

UNION ALL

-- 6. Trend period types
SELECT
    'TRENDS',
    'Invalid period_type',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.daily_weekly_trends
WHERE period_type NOT IN ('DAILY', 'WEEKLY')

UNION ALL

-- 7. Daily period validation
SELECT
    'TRENDS',
    'Invalid DAILY period',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.daily_weekly_trends
WHERE period_type = 'DAILY'
  AND order_date <> period_start

UNION ALL

-- 8. Weekly period validation
SELECT
    'TRENDS',
    'Invalid WEEKLY order_date',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.daily_weekly_trends
WHERE period_type = 'WEEKLY'
  AND order_date IS NOT NULL

UNION ALL

-- 9. Weekly Monday alignment
SELECT
    'TRENDS',
    'Weekly not starting Monday',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.daily_weekly_trends
WHERE period_type = 'WEEKLY'
  AND dayofweek(period_start) <> 2

UNION ALL

-- 10. Negative product revenue
SELECT
    'REVENUE',
    'Negative product revenue',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.sales_by_product
WHERE total_revenue < 0

UNION ALL

-- 11. Invalid product order count
SELECT
    'REVENUE',
    'Invalid product order count',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.sales_by_product
WHERE total_orders <= 0

UNION ALL

-- 12. Negative customer revenue
SELECT
    'REVENUE',
    'Negative customer revenue',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.revenue_by_customer
WHERE total_revenue < 0

UNION ALL

-- 13. Zero-order customer rules
SELECT
    'CUSTOMER',
    'Invalid zero-order customer values',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.revenue_by_customer
WHERE total_orders = 0
  AND (
        total_revenue <> 0
        OR avg_order_value IS NOT NULL
        OR lifetime_value_actual <> 0
      )

UNION ALL

-- 14. Average order value
SELECT
    'REVENUE',
    'Invalid avg_order_value',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.revenue_by_customer
WHERE total_orders > 0
  AND ABS(
        avg_order_value -
        ROUND(total_revenue / total_orders, 2)
      ) > 0.01

UNION ALL

-- 15. Lifetime value
SELECT
    'REVENUE',
    'Invalid lifetime_value_actual',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.revenue_by_customer
WHERE ABS(lifetime_value_actual - total_revenue) > 0.01

UNION ALL

-- 16. Product revenue reconciliation
SELECT
    'RECONCILIATION',
    'Product revenue',
    ROUND(
        (
            SELECT COALESCE(SUM(total_revenue), 0)
            FROM gold.sales_by_product
        )
        -
        (
            SELECT COALESCE(SUM(o.total_amount), 0)
            FROM silver.orders o
            INNER JOIN silver.products p
                ON o.product_id = p.product_id
            WHERE o.is_valid = true
              AND p.is_valid = true
              AND o.order_status = 'Completed'
        ),
        2
    ),
    0,
    CASE
        WHEN ABS(
            (
                SELECT COALESCE(SUM(total_revenue), 0)
                FROM gold.sales_by_product
            )
            -
            (
                SELECT COALESCE(SUM(o.total_amount), 0)
                FROM silver.orders o
                INNER JOIN silver.products p
                    ON o.product_id = p.product_id
                WHERE o.is_valid = true
                  AND p.is_valid = true
                  AND o.order_status = 'Completed'
            )
        ) <= 0.01
        THEN 'PASS'
        ELSE 'FAIL'
    END

UNION ALL

-- 17. Customer revenue reconciliation
SELECT
    'RECONCILIATION',
    'Customer revenue',
    ROUND(
        (
            SELECT COALESCE(SUM(total_revenue), 0)
            FROM gold.revenue_by_customer
        )
        -
        (
            SELECT COALESCE(SUM(o.total_amount), 0)
            FROM silver.orders o
            INNER JOIN silver.customers c
                ON o.customer_id = c.customer_id
            INNER JOIN silver.products p
                ON o.product_id = p.product_id
            WHERE o.is_valid = true
              AND c.is_valid = true
              AND p.is_valid = true
              AND o.order_status = 'Completed'
        ),
        2
    ),
    0,
    CASE
        WHEN ABS(
            (
                SELECT COALESCE(SUM(total_revenue), 0)
                FROM gold.revenue_by_customer
            )
            -
            (
                SELECT COALESCE(SUM(o.total_amount), 0)
                FROM silver.orders o
                INNER JOIN silver.customers c
                    ON o.customer_id = c.customer_id
                INNER JOIN silver.products p
                    ON o.product_id = p.product_id
                WHERE o.is_valid = true
                  AND c.is_valid = true
                  AND p.is_valid = true
                  AND o.order_status = 'Completed'
            )
        ) <= 0.01
        THEN 'PASS'
        ELSE 'FAIL'
    END

UNION ALL

-- 18. NULL product IDs
SELECT
    'SCHEMA',
    'NULL product_id',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.sales_by_product
WHERE product_id IS NULL

UNION ALL

-- 19. NULL customer IDs
SELECT
    'SCHEMA',
    'NULL customer_id',
    COUNT(*),
    0,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM gold.revenue_by_customer
WHERE customer_id IS NULL

UNION ALL

-- 20. Final table count
SELECT
    'SUMMARY',
    'Gold tables',
    4,
    4,
    'PASS';

validation,check_name,actual_value,expected_value,status
GRAIN,Duplicate product_id,0.00,0,PASS
GRAIN,Duplicate customer_id,0.00,0,PASS
SEGMENTATION,Invalid segment_type,0.00,0,PASS
SEGMENTATION,Customer count reconciliation,9940.00,9940,PASS
TRENDS,Invalid period_type,0.00,0,PASS
TRENDS,Invalid DAILY period,0.00,0,PASS
TRENDS,Invalid WEEKLY order_date,0.00,0,PASS
TRENDS,Weekly not starting Monday,0.00,0,PASS
REVENUE,Negative product revenue,0.00,0,PASS
REVENUE,Invalid product order count,0.00,0,PASS
